# Constrained GFS for astronomical target selection

This reproducible toy study asks whether the package's existing fixed-size GFS exchange can first repair a seeded Madow design and then improve it without changing the prescribed first-order inclusion probabilities (FIPs).

> **LP verifies existence and provides benchmarks; GFS constructs, repairs, navigates, and optimizes.**

In [ ]:
SETTINGS = {
    "root_seed": 12345,
    "repair_proposals": 20_000,
    "entropy_proposals": 20_000,
    "sip_proposals": 20_000,
    "checkpoint_every": 250,
    "invariant_tolerance": 1e-12,
    "sip_zero_tolerance": 1e-12,
    "write_outputs": False,
    "output_dir": "simulations/results/constrained_gfs_astronomy",
}

In [ ]:
from dataclasses import asdict
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from graphical_sampling.constrained import (
    ConstraintSystem, GroupCapacity, analyze_pair_attainability,
    attainable_pair_score, build_madow_design, enumerate_feasible_samples,
    incidence_matrix, is_feasible, run_constraint_repair,
    run_feasible_gfs_search, seeded_madow_order,
    solve_feasibility_reference, solve_numerical_maxent,
    validate_probability_mass,
)
from graphical_sampling.maxent.finite_population import design_probability_mass

plt.rcParams.update({"figure.figsize": (7.2, 4.4), "axes.grid": False})
study_started = perf_counter()

# 1. Motivation and the Madow-to-repair scientific question

The experiment separates existence, repair, feasible navigation, and optimization. Its proposed method is the inherited Algorithm-4 GFS segment exchange. During repair, an exchange is accepted only when it strictly decreases infeasible probability mass; after repair, every exchange that creates an infeasible sample is rejected. No projection, FIP repair, or LP-derived search state is used.

# 2. Population and prescribed FIPs

Root seed `12345` is the **predeclared reproducible demonstration seed**. The permutation below is generated solely from its role-0 stream, before and without any constraint or objective input. The random-order demonstration was selected after the original population order proved constraint-feasible, so a methodological paper must repeat this analysis over many independently predeclared seeds rather than describe this one as unselected.

In [ ]:
N, n = 10, 4
PI = np.array([0.55, 0.45, 0.50, 0.40, 0.35, 0.35, 0.40, 0.35, 0.35, 0.30])
DISPLAY_COORDINATES = np.array([
    [0.08, 0.78], [0.20, 0.70], [0.34, 0.86], [0.45, 0.73], [0.61, 0.83],
    [0.73, 0.69], [0.18, 0.29], [0.38, 0.18], [0.59, 0.31], [0.82, 0.20],
], dtype=float)
assert np.isclose(PI.sum(), n, atol=1e-12, rtol=0.0)
# This line deliberately precedes construction of every hard constraint.
madow_order = seeded_madow_order(N, SETTINGS["root_seed"])
print(f"sum(pi)={PI.sum():.12f}; seeded Madow order={madow_order.tolist()}")
display(pd.DataFrame({"unit": np.arange(N), "pi": PI, "x": DISPLAY_COORDINATES[:, 0], "y": DISPLAY_COORDINATES[:, 1]}))

# 3. Artificial fibre constraints

Three pairs represent artificial fibre collisions. A fourth rule permits at most two targets from group `{6, 7, 8}`. These constraints are illustrative rather than a survey instrument model.

In [ ]:
EXPLICIT_FORBIDDEN = ((0, 1), (2, 3), (4, 5))
constraints = ConstraintSystem(
    population_size=N, sample_size=n,
    forbidden_pairs=frozenset(frozenset(pair) for pair in EXPLICIT_FORBIDDEN),
    group_capacities=(GroupCapacity(frozenset({6, 7, 8}), 2),),
)
fig_target, ax = plt.subplots()
ax.scatter(DISPLAY_COORDINATES[:, 0], DISPLAY_COORDINATES[:, 1], s=80, c=PI, cmap="viridis")
for unit, (x, y) in enumerate(DISPLAY_COORDINATES):
    ax.annotate(str(unit), (x, y), xytext=(5, 5), textcoords="offset points")
for first, second in EXPLICIT_FORBIDDEN:
    points = DISPLAY_COORDINATES[[first, second]]
    ax.plot(points[:, 0], points[:, 1], "--", color="crimson", linewidth=1.5)
ax.set(title="Artificial target field and explicit forbidden pairs", xlabel="display x", ylabel="display y")
fig_target.tight_layout()
plt.show()

# 4. Initial Madow/GFS design

The initial state comes from `Design._build` through the package's Madow/GFS construction in the role-0 order. Coordinates serve only to display the toy targets; the supplied order fixes the one-zone construction.

In [ ]:
madow_design = build_madow_design(PI, DISPLAY_COORDINATES, madow_order)
madow_mass = design_probability_mass(madow_design)
madow_diag = validate_probability_mass(
    madow_mass, PI, constraints, require_feasible=False,
    tolerance=SETTINGS["invariant_tolerance"],
)
print(
    f"Madow support={madow_diag.support_size}, entropy={madow_diag.entropy:.8f}, "
    f"max FIP error={madow_diag.max_fip_error:.3e}"
)
np.testing.assert_allclose(madow_diag.fip, PI, atol=SETTINGS["invariant_tolerance"], rtol=0.0)

# 5. Initial violations and infeasible mass

For a design `D`, `V(D)` is the total probability assigned to hard-constraint-infeasible samples. The initial Madow design is deliberately not conditioned on the artificial constraints.

In [ ]:
violations = pd.DataFrame([
    {"sample": sample, "probability": probability}
    for sample, probability in sorted(madow_mass.items())
    if probability > SETTINGS["invariant_tolerance"] and not is_feasible(sample, constraints)
])
print(
    f"Initial violating samples={madow_diag.violating_support_size}; "
    f"V(D0)={madow_diag.infeasible_mass:.12f}"
)
display(violations)

# 6. GFS constraint repair

Stage 1 proposes ordinary inherited GFS exchanges with maximal exchanged mass and accepts only strict decreases in `V(D)`. Infeasible support may remain temporarily; accepted mass is never deleted or renormalized.

In [ ]:
repair_result = run_constraint_repair(
    madow_design, PI, constraints,
    proposals=SETTINGS["repair_proposals"],
    seed=np.random.SeedSequence([SETTINGS["root_seed"], 1]),
    checkpoint_every=SETTINGS["checkpoint_every"],
    tolerance=SETTINGS["invariant_tolerance"],
)
repaired_mass = design_probability_mass(repair_result.final_design)
repaired_diag = validate_probability_mass(
    repaired_mass, PI, constraints, require_feasible=repair_result.reached_feasibility,
    tolerance=SETTINGS["invariant_tolerance"],
)
repair_trace = pd.DataFrame([asdict(row) for row in repair_result.trace])
fig_repair, ax = plt.subplots()
ax.plot(repair_trace["proposal"], repair_trace["infeasible_mass"], color="tab:red", label="infeasible mass")
ax.set(xlabel="proposed GFS exchanges", ylabel="V(D)", title="Strict GFS constraint repair")
secondary = ax.twinx()
secondary.plot(repair_trace["proposal"], repair_trace["accepted"], color="tab:blue", alpha=0.45, label="accepted")
secondary.set_ylabel("cumulative accepted exchanges")
fig_repair.tight_layout()
plt.show()
print(repair_result.status, asdict(repair_result.counts), f"runtime={repair_result.runtime_seconds:.2f}s")

# 7. Feasible GFS entropy optimization

Stage 2 starts only from the repaired GFS state. Its role-2 stream is independent of permutation, repair, and SIP streams. An independent per-pair LP mask is computed here only to define the later SIP score; its benchmark table is deferred to Section 9.

In [ ]:
feasible_samples = enumerate_feasible_samples(constraints)
incidence = incidence_matrix(feasible_samples, N)
pair_analysis = analyze_pair_attainability(PI, feasible_samples, incidence, constraints)
attainable_pairs = tuple(sorted(pair_analysis.marginal_attainable))

if repair_result.reached_feasibility:
    entropy_result = run_feasible_gfs_search(
        repair_result.final_design, PI, constraints, objective="entropy",
        proposals=SETTINGS["entropy_proposals"],
        seed=np.random.SeedSequence([SETTINGS["root_seed"], 2]),
        checkpoint_every=SETTINGS["checkpoint_every"],
        tolerance=SETTINGS["invariant_tolerance"],
    )
    entropy_mass = design_probability_mass(entropy_result.final_design)
    entropy_diag = validate_probability_mass(
        entropy_mass, PI, constraints, require_feasible=True,
        tolerance=SETTINGS["invariant_tolerance"],
    )
else:
    entropy_result = None
    entropy_mass, entropy_diag = repaired_mass, repaired_diag

entropy_trace = pd.DataFrame([asdict(row) for row in entropy_result.trace]) if entropy_result else pd.DataFrame()
fig_entropy, ax = plt.subplots()
if entropy_result:
    ax.plot(entropy_trace["accepted"], entropy_trace["entropy"], color="tab:green")
else:
    ax.text(0.5, 0.5, "Repair failed; Stage 2 skipped", ha="center", va="center")
ax.set(xlabel="accepted entropy exchanges", ylabel="entropy", title="Feasible GFS entropy hill climb")
fig_entropy.tight_layout()
plt.show()

# 8. Feasible GFS pairwise optimization

The role-3 run restarts independently from the repaired GFS design. It scores only marginal-attainable unordered pairs and rejects every hard-infeasible proposal before evaluating the score.

In [ ]:
if repair_result.reached_feasibility:
    sip_result = run_feasible_gfs_search(
        repair_result.final_design, PI, constraints, objective="sip",
        attainable_pairs=attainable_pairs, proposals=SETTINGS["sip_proposals"],
        seed=np.random.SeedSequence([SETTINGS["root_seed"], 3]),
        checkpoint_every=SETTINGS["checkpoint_every"],
        tolerance=SETTINGS["invariant_tolerance"],
    )
    sip_mass = design_probability_mass(sip_result.final_design)
    sip_diag = validate_probability_mass(
        sip_mass, PI, constraints, require_feasible=True,
        tolerance=SETTINGS["invariant_tolerance"],
    )
else:
    sip_result = None
    sip_mass, sip_diag = repaired_mass, repaired_diag

attainable_before = np.array([repaired_diag.sip[pair] for pair in attainable_pairs])
attainable_after = np.array([sip_diag.sip[pair] for pair in attainable_pairs])
print(
    "Attainable zero SIPs:",
    int(np.count_nonzero(attainable_before <= SETTINGS["sip_zero_tolerance"])),
    "->", int(np.count_nonzero(attainable_after <= SETTINGS["sip_zero_tolerance"])),
)

# 9. Independent LP existence and benchmark results

The LP below neither initializes nor repairs a GFS design. It verifies existence over all enumerated hard-feasible samples and supplies per-pair attainability limits. The numerical MaxEnt result is a global convex reference up to its reported numerical tolerance, not a GFS result.

In [ ]:
lp_result = solve_feasibility_reference(PI, feasible_samples, incidence)
maxent_reference = solve_numerical_maxent(PI, feasible_samples, incidence)
category_table = pd.DataFrame([
    {"pair": pair, "category": category, "maximum_SIP": pair_analysis.maximum_sip[pair]}
    for pair, category in pair_analysis.categories.items()
])
display(category_table)
print(asdict(lp_result))
print(maxent_reference.label, f"entropy={maxent_reference.entropy:.8f}", f"error={maxent_reference.max_equality_error:.3e}")

# 10. Results and numerical MaxEnt comparison

The table and common-scale heatmaps compare the original Madow state, the repaired GFS state, the two independent Stage-2 results, and the numerical reference. Crosses mark hard constraint-incompatible cells.

In [ ]:
maxent_mass = {sample: float(probability) for sample, probability in zip(feasible_samples, maxent_reference.probabilities)}
maxent_diag = validate_probability_mass(maxent_mass, PI, constraints, require_feasible=True, tolerance=1e-9)

constraint_incompatible = set(pair_analysis.explicit_forbidden) | set(pair_analysis.other_constraint_incompatible)
vmax = max(float(np.max(madow_diag.sip)), float(np.max(repaired_diag.sip)), float(np.max(sip_diag.sip)))
fig_sip, axes = plt.subplots(1, 3, figsize=(13.2, 3.9), constrained_layout=True)
for axis, matrix, title in zip(
    axes, [madow_diag.sip, repaired_diag.sip, sip_diag.sip],
    ["Initial Madow", "Repaired GFS", "Pair-optimized GFS"],
):
    image = axis.imshow(matrix, vmin=0.0, vmax=vmax, cmap="magma")
    for first, second in constraint_incompatible:
        axis.text(second, first, "×", ha="center", va="center", color="cyan", fontsize=11)
        axis.text(first, second, "×", ha="center", va="center", color="cyan", fontsize=11)
    axis.set(title=title, xlabel="unit", ylabel="unit")
fig_sip.colorbar(image, ax=axes, shrink=0.82, label="second-order inclusion probability")
plt.show()

zero_tol = SETTINGS["sip_zero_tolerance"]
maxent_attainable = np.array([maxent_diag.sip[pair] for pair in attainable_pairs])
comparison = pd.DataFrame([
    {"design": "Initial Madow", "support": madow_diag.support_size, "entropy": madow_diag.entropy, "V(D)": madow_diag.infeasible_mass, "max FIP error": madow_diag.max_fip_error, "attainable zero SIPs": int(np.count_nonzero(np.array([madow_diag.sip[p] for p in attainable_pairs]) <= zero_tol))},
    {"design": "Repaired GFS", "support": repaired_diag.support_size, "entropy": repaired_diag.entropy, "V(D)": repaired_diag.infeasible_mass, "max FIP error": repaired_diag.max_fip_error, "attainable zero SIPs": int(np.count_nonzero(attainable_before <= zero_tol))},
    {"design": "Entropy GFS", "support": entropy_diag.support_size, "entropy": entropy_diag.entropy, "V(D)": entropy_diag.infeasible_mass, "max FIP error": entropy_diag.max_fip_error, "attainable zero SIPs": int(np.count_nonzero(np.array([entropy_diag.sip[p] for p in attainable_pairs]) <= zero_tol))},
    {"design": "Pair GFS", "support": sip_diag.support_size, "entropy": sip_diag.entropy, "V(D)": sip_diag.infeasible_mass, "max FIP error": sip_diag.max_fip_error, "attainable zero SIPs": int(np.count_nonzero(attainable_after <= zero_tol))},
    {"design": maxent_reference.label, "support": maxent_diag.support_size, "entropy": maxent_diag.entropy, "V(D)": maxent_diag.infeasible_mass, "max FIP error": maxent_diag.max_fip_error, "attainable zero SIPs": int(np.count_nonzero(maxent_attainable <= zero_tol))},
])
counts_table = pd.DataFrame([
    {"phase": "repair", **asdict(repair_result.counts)},
    {"phase": "entropy", **(asdict(entropy_result.counts) if entropy_result else {})},
    {"phase": "sip", **(asdict(sip_result.counts) if sip_result else {})},
])
display(comparison)
display(counts_table)

if repair_result.reached_feasibility:
    np.testing.assert_allclose(repaired_diag.fip, madow_diag.fip, atol=1e-10, rtol=0.0)
    assert np.linalg.norm(repaired_diag.sip - madow_diag.sip) > 0.0
if entropy_result and entropy_result.counts.accepted:
    np.testing.assert_allclose(entropy_diag.fip, repaired_diag.fip, atol=1e-10, rtol=0.0)
    assert np.linalg.norm(entropy_diag.sip - repaired_diag.sip) > 0.0
if sip_result and sip_result.counts.accepted:
    np.testing.assert_allclose(sip_diag.fip, repaired_diag.fip, atol=1e-10, rtol=0.0)
    assert np.linalg.norm(sip_diag.sip - repaired_diag.sip) > 0.0

repair_counts = asdict(repair_result.counts)
entropy_counts = asdict(entropy_result.counts) if entropy_result else {name: 0 for name in repair_counts}
sip_counts = asdict(sip_result.counts) if sip_result else {name: 0 for name in repair_counts}
results = {
    "settings": dict(SETTINGS),
    "madow": {"order": madow_order.copy(), "support_size": madow_diag.support_size, "entropy": madow_diag.entropy, "infeasible_mass": madow_diag.infeasible_mass, "violating_support_size": madow_diag.violating_support_size, "max_fip_error": madow_diag.max_fip_error, "fip": madow_diag.fip, "sip": madow_diag.sip},
    "repair": {"status": repair_result.status, "reached_feasibility": repair_result.reached_feasibility, "counts": repair_counts, "runtime_seconds": repair_result.runtime_seconds, "final_infeasible_mass": repaired_diag.infeasible_mass, "max_fip_error": repaired_diag.max_fip_error, "support_changed": madow_mass != repaired_mass, "support_shared": len(set(madow_mass) & set(repaired_mass)), "support_added": len(set(repaired_mass) - set(madow_mass)), "support_removed": len(set(madow_mass) - set(repaired_mass)), "sip_change_norm": float(np.linalg.norm(repaired_diag.sip - madow_diag.sip)), "all_support_feasible": all(is_feasible(sample, constraints) for sample, probability in repaired_mass.items() if probability > SETTINGS["invariant_tolerance"]), "trace": repair_trace},
    "entropy": {"status": entropy_result.status if entropy_result else "skipped", "counts": entropy_counts, "runtime_seconds": entropy_result.runtime_seconds if entropy_result else 0.0, "entropy_before": repaired_diag.entropy, "entropy_after": entropy_diag.entropy, "entropy_gap_to_maxent": maxent_reference.entropy - entropy_diag.entropy, "max_fip_error": entropy_diag.max_fip_error, "support_shared": len(set(repaired_mass) & set(entropy_mass)), "support_added": len(set(entropy_mass) - set(repaired_mass)), "support_removed": len(set(repaired_mass) - set(entropy_mass)), "sip_change_norm": float(np.linalg.norm(entropy_diag.sip - repaired_diag.sip)), "all_support_feasible": all(is_feasible(sample, constraints) for sample, probability in entropy_mass.items() if probability > SETTINGS["invariant_tolerance"]), "trace": entropy_trace},
    "sip": {"status": sip_result.status if sip_result else "skipped", "counts": sip_counts, "runtime_seconds": sip_result.runtime_seconds if sip_result else 0.0, "score_before": attainable_pair_score(repaired_mass, attainable_pairs), "score_after": attainable_pair_score(sip_mass, attainable_pairs), "attainable_zero_before": int(np.count_nonzero(attainable_before <= zero_tol)), "attainable_zero_after": int(np.count_nonzero(attainable_after <= zero_tol)), "attainable_min_before": float(np.min(attainable_before)), "attainable_min_after": float(np.min(attainable_after)), "attainable_median_before": float(np.median(attainable_before)), "attainable_median_after": float(np.median(attainable_after)), "attainable_mean_before": float(np.mean(attainable_before)), "attainable_mean_after": float(np.mean(attainable_after)), "max_fip_error": sip_diag.max_fip_error, "support_shared": len(set(repaired_mass) & set(sip_mass)), "support_added": len(set(sip_mass) - set(repaired_mass)), "support_removed": len(set(repaired_mass) - set(sip_mass)), "sip_change_norm": float(np.linalg.norm(sip_diag.sip - repaired_diag.sip)), "all_support_feasible": all(is_feasible(sample, constraints) for sample, probability in sip_mass.items() if probability > SETTINGS["invariant_tolerance"])},
    "lp_feasibility": asdict(lp_result),
    "pair_attainability": {"explicit_forbidden_count": len(pair_analysis.explicit_forbidden), "other_constraint_incompatible_count": len(pair_analysis.other_constraint_incompatible), "marginal_inattainable_count": len(pair_analysis.marginal_inattainable), "marginal_attainable_count": len(pair_analysis.marginal_attainable), "categories": dict(pair_analysis.categories), "maximum_sip": pair_analysis.maximum_sip},
    "maxent_reference": {"label": maxent_reference.label, "success": maxent_reference.success, "entropy": maxent_reference.entropy, "max_equality_error": maxent_reference.max_equality_error},
    "comparison": comparison,
    "figures": {"target_field", "repair_trace", "entropy_trace", "sip_heatmaps"},
    "runtime_seconds": float(perf_counter() - study_started),
}

if SETTINGS["write_outputs"]:
    output_dir = Path(SETTINGS["output_dir"])
    output_dir.mkdir(parents=True, exist_ok=True)
    fig_target.savefig(output_dir / "target_field.png", dpi=180, bbox_inches="tight")
    fig_repair.savefig(output_dir / "repair_trace.png", dpi=180, bbox_inches="tight")
    fig_entropy.savefig(output_dir / "entropy_trace.png", dpi=180, bbox_inches="tight")
    fig_sip.savefig(output_dir / "sip_heatmaps.png", dpi=180, bbox_inches="tight")
    comparison.to_csv(output_dir / "comparison.csv", index=False)
    counts_table.to_csv(output_dir / "proposal_counts.csv", index=False)

# 11. Short conclusions and limitations

The numerical evidence must be interpreted in the order tested: GFS constructs the seeded Madow state, GFS exchanges repair it, and feasible GFS exchanges navigate and optimize it; LP only verifies existence and supplies reference limits. Success for this one predeclared demonstration seed does not establish universal repair, connectivity of the feasible move graph, global hill-climbing optimality, or DESI-scale practicality. Low acceptance, stagnation, remaining attainable zero-SIP pairs, or a gap to the numerical MaxEnt benchmark are informative outcomes rather than reasons to substitute an LP state.